**Cell #01**

# RAG11 Nutrition — Stage 3: Hybrid Search (Dense + Keyword) Examples

Adds **hybrid search** on top of the Stage 2 retrieval + generation pipeline:
dense (vector) search and keyword (full-text) search run side by side, then
their ranked lists are merged with **Reciprocal Rank Fusion (RRF)** — and
demonstrates it with 3 nutrition-specific examples chosen to show where each
method wins (and where fusing them beats either alone).

## The problem, in one picture

Dense (vector) search only compares "meaning vectors." It's great at "this is
roughly about the same topic," but bad at "this contains the exact number/word
I need." Say a nutrition RAG gets asked *"How many grams of protein per
kilogram of body weight does the RDA recommend?"*:

- Pure vector search might return chunks that are all about protein — muscle
  building, protein sources, amino acids — but the one chunk that literally
  says **"0.8 g/kg/day"** might rank 6th, because a paragraph about "protein
  timing for athletes" sounds more similar overall.
- A keyword search for **"0.8"** or **"RDA"** would instantly find the exact
  chunk, because it's just matching text, not meaning.
- **Hybrid** = do both searches, then combine the two ranked lists (e.g.
  "chunk A was #1 in vector search and #3 in keyword search → give it a
  combined score"). This catches cases where the right chunk scores well on
  one method but not the other.

Think of it like: vector search = *"find me something in the same
neighborhood,"* keyword search = *"find me the exact house number."*

## How the fusion works — Reciprocal Rank Fusion (RRF)

```
rrf_score(chunk) = sum over methods of  1 / (k + rank_method(chunk))      k = 60
```

Each chunk's combined score only depends on its *position* in each ranked
list — never on comparing cosine distance to `ts_rank_cd` directly (they're
not on the same scale). A chunk absent from one list simply contributes 0 for
it, so a chunk either method ranks highly still surfaces, and a chunk both
methods agree on rises to the top.

## Prerequisites

Same as `stage2_ask_examples2_rerank.ipynb`, plus **one additive SQL
migration**: `sql/create_sql_tables.sql` now also creates a generated
`chunk_tsv` column, a GIN index, and the `match_rag11_child_chunks_keyword`
RPC — re-run it once in the Supabase SQL Editor before this notebook (it's
idempotent; existing rows/embeddings are untouched, `chunk_tsv` just
backfills for free from the `rowJSON` payload that already exists --
it reads `rowJSON->>'text'` directly, not `chunk_text`, since a Postgres
generated column can't reference another generated column).

This notebook gets its retrieval/generation logic from `./reusable_code/`
— the same package `stage2_ask_examples1.ipynb` and
`stage2_ask_examples2_rerank.ipynb` import from, so all three notebooks share
one implementation of `ask_question` (`reusable_code/generation.py`),
`retrieve_chunks` (`reusable_code/retrieval.py`), and `hybrid_search`
(`reusable_code/hybrid_search.py`).
See `documentation/HOW_IT_WORKS_Hybrid_Search.html` for the full write-up and
`reusable_code/README.md` for the migration note.


In [1]:
from reusable_code import (
    init_clients,
    ask_question,
    retrieve_chunks,
    retrieve_chunks_keyword,
    reciprocal_rank_fusion,
    hybrid_search,
    rerank_chunks,
    NUM_CONTEXT_CHUNKS,
    RRF_K,
    EMBEDDING_MODEL,
    RERANK_MODEL,
    GENERATION_MODEL,
)

clients = init_clients()
print("Clients ready.")
print("Embedding model:", EMBEDDING_MODEL, "| Rerank model:", RERANK_MODEL,
      "| Generation model:", GENERATION_MODEL, "| RRF k:", RRF_K)


Clients ready.
Embedding model: voyage-3 | Rerank model: rerank-2 | Generation model: claude-sonnet-5 | RRF k: 60


**Cell #03**

## A helper to see dense-only vs. keyword-only vs. hybrid side by side

`show_hybrid_comparison()` runs all three against the real database: a plain
dense search, a plain keyword search, and `hybrid_search()` (which fuses
both via RRF) — printed together so the effect of fusion is visible rather
than assumed. It also flags every chunk that only one of the two raw methods
found at all, since that's exactly the case hybrid search exists to catch.


In [2]:
def show_hybrid_comparison(question: str, pool: int = 15, top_n: int = NUM_CONTEXT_CHUNKS):
    """Run dense search, keyword search, and hybrid_search() (RRF-fused) for
    `question` and print all three side by side. Returns
    (dense, keyword, fused) so the caller can inspect or reuse any of them."""
    print(f"Q: {question}\n")

    dense = retrieve_chunks(question, match_count=pool)
    print(f"-- Dense search top {len(dense)} (cosine distance, closest first) --")
    for i, row in enumerate(dense, start=1):
        preview = row["rowJSON"]["text"].replace("\n", " ")[:100]
        source = row["rowJSON"].get("source_key", "?")
        print(f"  {i:>2}. dist={row['cosine_distance']:.4f}  [{source}]  {preview}...")

    keyword = retrieve_chunks_keyword(question, match_count=pool)
    print(f"\n-- Keyword search top {len(keyword)} (ts_rank_cd, best first) --")
    if not keyword:
        print("  (no full-text matches at all -- question shares no indexed vocabulary with any chunk)")
    for i, row in enumerate(keyword, start=1):
        preview = row["rowJSON"]["text"].replace("\n", " ")[:100]
        source = row["rowJSON"].get("source_key", "?")
        print(f"  {i:>2}. rank={row['text_rank']:.4f}  [{source}]  {preview}...")

    fused = hybrid_search(question, match_count=top_n, dense_pool=pool, keyword_pool=pool)
    print(f"\n-- Hybrid (RRF-fused) top {len(fused)} --")
    for i, row in enumerate(fused, start=1):
        preview = row["rowJSON"]["text"].replace("\n", " ")[:100]
        source = row["rowJSON"].get("source_key", "?")
        dr = row["dense_rank"] if row["dense_rank"] is not None else "-"
        kr = row["keyword_rank"] if row["keyword_rank"] is not None else "-"
        both = " <- found by BOTH methods" if row["dense_rank"] and row["keyword_rank"] else ""
        print(f"  {i:>2}. rrf={row['rrf_score']:.4f}  dense=#{dr:<3} keyword=#{kr:<3}  [{source}]  {preview}...{both}")

    dense_only_top1 = bool(dense) and (not keyword or dense[0]["rowGUID"] not in {r["rowGUID"] for r in keyword})
    keyword_found_something_dense_missed = bool(keyword) and bool(dense) and \
        any(r["rowGUID"] not in {d["rowGUID"] for d in dense[:top_n]} for r in keyword[:top_n])
    print(f"\nKeyword search surfaced a top-{top_n} chunk dense search alone would have missed: "
          f"{keyword_found_something_dense_missed}")
    print("-" * 80)
    return dense, keyword, fused


example_results = {}   # question -> (dense, keyword, fused), filled in by the 3 examples below


**Cell #05**

## Three nutrition examples for the hybrid-search demonstration

Each chosen for a different reason dense-only or keyword-only search
struggles with on its own:

1. **A precise number buried among broadly-similar chunks** — the running
   example above: many chunks are generally about protein; only one has the
   actual RDA figure. Keyword search finds it instantly; dense search alone
   buries it.
2. **An exact acronym/technical term** — `PDCAAS` is a specific scoring
   method name. Dense search treats it as "generically about protein
   quality"; keyword search matches the literal term.
3. **A conceptual, paraphrased question with little exact vocabulary
   overlap** — the flip side: here keyword search is expected to return
   few or no hits (nothing to match on literally), so the fused ranking
   should fall back gracefully to dense search's ordering rather than
   losing relevant chunks.


In [3]:
EXAMPLE_QUESTIONS = [
    # 1. Numeric fact vs. general topic chunks -- keyword search should win.
    "How many grams of protein per kilogram of body weight does the RDA recommend for an average adult?",
    # 2. Exact acronym/technical term -- keyword search should find it directly.
    "What does the PDCAAS score measure when evaluating a protein source?",
    # 3. Conceptual/paraphrased question, little literal vocabulary overlap --
    #    keyword search should come back sparse; hybrid should still work.
    "If someone eats plenty of calories but the wrong balance of foods, can they still end up malnourished?",
]


**Cell #07**

### Example 1 — a precise number among many general-topic chunks


In [4]:
example_results[EXAMPLE_QUESTIONS[0]] = show_hybrid_comparison(EXAMPLE_QUESTIONS[0])


Q: How many grams of protein per kilogram of body weight does the RDA recommend for an average adult?

-- Dense search top 15 (cosine distance, closest first) --
   1. dist=0.3621  [source11]  [Source: _OceanofPDF.com_Nancy_Clarks_Sports_Nutrition_Guidebook_-_Nancy_Clark.pdf | Section: Protei...
   2. dist=0.3739  [source3]  [Source: Nutrition-Science-and-Everyday-Application-1773787282.pdf | Section: Protein in Foods and D...
   3. dist=0.3799  [source11]  [Source: _OceanofPDF.com_Nancy_Clarks_Sports_Nutrition_Guidebook_-_Nancy_Clark.pdf | Section: Protei...
   4. dist=0.4053  [source11]  [Source: _OceanofPDF.com_Nancy_Clarks_Sports_Nutrition_Guidebook_-_Nancy_Clark.pdf | Section: Protei...
   5. dist=0.4168  [source11]  [Source: _OceanofPDF.com_Nancy_Clarks_Sports_Nutrition_Guidebook_-_Nancy_Clark.pdf | Section: Protei...
   6. dist=0.4172  [source17]  [Source: _OceanofPDF.com_Encyclopedia_of_foods_-_Mayo_Clinic.pdf | Section: Chapter 2. The Nutrients...
   7. dist=0.4184  [source3] 

**Cell #09**

### Example 2 — an exact acronym/technical term


In [5]:
example_results[EXAMPLE_QUESTIONS[1]] = show_hybrid_comparison(EXAMPLE_QUESTIONS[1])


Q: What does the PDCAAS score measure when evaluating a protein source?

-- Dense search top 15 (cosine distance, closest first) --
   1. dist=0.4904  [source4]  [Source: Advanced_Nutrition_and_Human_Metabolism.pdf | Section: Ch 6: Protein | Pages 35-37]  Chapte...
   2. dist=0.5882  [source11]  [Source: _OceanofPDF.com_Nancy_Clarks_Sports_Nutrition_Guidebook_-_Nancy_Clark.pdf | Section: Protei...
   3. dist=0.5971  [source4]  [Source: Advanced_Nutrition_and_Human_Metabolism.pdf | Section: Ch 6: Protein | Pages 35-37]  . Grop...
   4. dist=0.6000  [source16]  [Source: Encyclopedia_of_Human_Nutrition_4th_Edition_Benjamin_Caballero.pdf | Section: Amino acids: ...
   5. dist=0.6004  [source16]  [Source: Encyclopedia_of_Human_Nutrition_4th_Edition_Benjamin_Caballero.pdf | Section: Amino acids: ...
   6. dist=0.6047  [source3]  [Source: Nutrition-Science-and-Everyday-Application-1773787282.pdf | Section: Introduction to Protei...
   7. dist=0.6064  [source16]  [Source: Encyclopedia_of_Human

**Cell #11**

### Example 3 — a conceptual question, little literal overlap


In [6]:
example_results[EXAMPLE_QUESTIONS[2]] = show_hybrid_comparison(EXAMPLE_QUESTIONS[2])


Q: If someone eats plenty of calories but the wrong balance of foods, can they still end up malnourished?

-- Dense search top 15 (cosine distance, closest first) --
   1. dist=0.4394  [source13]  [Source: vdoc.pub_medical-nutrition-and-disease-a-case-based-approach.pdf | Section: 1: Overview of ...
   2. dist=0.4468  [source11]  [Source: _OceanofPDF.com_Nancy_Clarks_Sports_Nutrition_Guidebook_-_Nancy_Clark.pdf | Section: Dietin...
   3. dist=0.4619  [source1]  [Source: human-nutrition-text.pdf | Section: Achieving a Healthy Diet | Pages 72-72]  Achieving a He...
   4. dist=0.4636  [source13]  [Source: vdoc.pub_medical-nutrition-and-disease-a-case-based-approach.pdf | Section: 1: Overview of ...
   5. dist=0.4693  [source13]  [Source: vdoc.pub_medical-nutrition-and-disease-a-case-based-approach.pdf | Section: 1: Overview of ...
   6. dist=0.4731  [source15]  [Source: vdoc.pub_nutrition-therapy-and-pathophysiology-2nd-edition.pdf | Section: 2 The Nutrition C...
   7. dist=0.4733  [sourc

**Cell #13**

## `ask_question(..., use_hybrid=...)` end to end

`use_hybrid` is an optional keyword argument on `ask_question` — it defaults
to `False`, so every existing call in `stage2_ask_examples1.ipynb`
(`ask_question(question)`) behaves exactly as before. Passing
`use_hybrid=True` swaps in `hybrid_search()` as the source of candidate
chunks instead of plain `retrieve_chunks()`. It also **composes** with
`use_rerank=True` from Stage 3's rerank notebook: hybrid search picks the
candidate pool, then Voyage's reranker re-scores that pool — the two
retrieval upgrades stack rather than compete.


In [ ]:
demo_question = EXAMPLE_QUESTIONS[0]

baseline = ask_question(demo_question, match_count=NUM_CONTEXT_CHUNKS, use_hybrid=False)
hybrid_answer = ask_question(demo_question, match_count=NUM_CONTEXT_CHUNKS, use_hybrid=True)
hybrid_reranked_answer = ask_question(demo_question, match_count=NUM_CONTEXT_CHUNKS,
                                       use_hybrid=True, use_rerank=True)

print("Q:", demo_question)

print("\n-- ask_question(..., use_hybrid=False) [default, dense-only] --")
print("chunks used:", baseline["chunks_used"], "| candidates considered:", baseline["candidates_considered"])
print("source pages:", baseline["source_pages"])
if baseline["short_answer"]:
    print("Short answer:", baseline["short_answer"])
print(baseline["answer"])

print("\n-- ask_question(..., use_hybrid=True) --")
print("chunks used:", hybrid_answer["chunks_used"], "| candidates considered:", hybrid_answer["candidates_considered"])
print("source pages:", hybrid_answer["source_pages"])
if hybrid_answer["short_answer"]:
    print("Short answer:", hybrid_answer["short_answer"])
print(hybrid_answer["answer"])

print("\n-- ask_question(..., use_hybrid=True, use_rerank=True) --")
print("chunks used:", hybrid_reranked_answer["chunks_used"],
      "| candidates considered:", hybrid_reranked_answer["candidates_considered"])
print("rerank model:", hybrid_reranked_answer["rerank_model"])
print("source pages:", hybrid_reranked_answer["source_pages"])
if hybrid_reranked_answer["short_answer"]:
    print("Short answer:", hybrid_reranked_answer["short_answer"])
print(hybrid_reranked_answer["answer"])


**Cell #15**

## Reciprocal Rank Fusion in isolation

`reciprocal_rank_fusion()` is a small, generic function — it doesn't know
anything about dense vs. keyword search specifically, just "several ranked
lists of dicts sharing a key." Worth seeing on its own, with hand-built
toy lists, so the k=60 damping and the "present in one list only" behavior
are visible without any network calls.


In [ ]:
toy_dense = [
    {"rowGUID": "chunkA", "note": "broadly about protein, ranked #1 by embedding similarity"},
    {"rowGUID": "chunkB", "note": "somewhat about protein, ranked #2"},
    {"rowGUID": "chunkC", "note": "the exact RDA figure, but ranked only #3 by embedding similarity"},
]
toy_keyword = [
    {"rowGUID": "chunkC", "note": "the exact RDA figure -- ranked #1 by keyword match"},
    {"rowGUID": "chunkD", "note": "a table mentioning the same figure, ranked #2 by keyword match"},
]

toy_fused = reciprocal_rank_fusion({"dense": toy_dense, "keyword": toy_keyword})
print(f"{'rank':<5}{'chunk':<8}{'rrf_score':<12}{'dense_rank':<12}{'keyword_rank':<14}note")
for i, row in enumerate(toy_fused, start=1):
    dr = row["dense_rank"] if row["dense_rank"] is not None else "-"
    kr = row["keyword_rank"] if row["keyword_rank"] is not None else "-"
    print(f"{i:<5}{row['rowGUID']:<8}{row['rrf_score']:<12.4f}{str(dr):<12}{str(kr):<14}{row['note']}")

print("\nchunkC (present in both lists, #3 dense / #1 keyword) out-ranks chunkA (#1 dense-only):",
      toy_fused[0]["rowGUID"] == "chunkC")


**Cell #17**

## Summary across the 3 examples

Same idea as the rerank notebook's summary table: a quick scan of how many
raw candidates each method returned, and whether keyword search surfaced
something dense search alone would have missed within the top-N.


In [ ]:
print(f"{'#':<3} {'dense hits':<11} {'keyword hits':<13} {'kw found dense missed?':<24} question")
for i, question in enumerate(EXAMPLE_QUESTIONS, start=1):
    dense, keyword, fused = example_results[question]
    top_n = NUM_CONTEXT_CHUNKS
    dense_top_ids = {r["rowGUID"] for r in dense[:top_n]}
    kw_found_something_new = bool(keyword) and any(r["rowGUID"] not in dense_top_ids for r in keyword[:top_n])
    print(f"{i:<3} {len(dense):<11} {len(keyword):<13} {str(kw_found_something_new):<24} {question}")


**Cell #19**

## Save workspace to GitHub

Synchronize this notebook and any code changes to GitHub (with auto lock
recovery and conflict resolution), the same helper `stage2_ask_examples2_rerank.ipynb`
uses -- shared via `reusable_code.save_to_github`.


In [ ]:
from reusable_code import save_to_github

save_to_github("stage2_ask_examples3_hybrid_search.ipynb - hybrid search examples added")
